In [ ]:
SPEC={'baseline_manifest': {'dataset_version': 'v1', 'dataset_built_at': '2026-10-21T15:56:12+00:00', 'clean_version': 'clean-v1', 'chunk_params': '350/50/40', 'id_scheme': '{text_sha256[:16]}-{order}', 'excluded_ids': 118, 'hash_groups': 1023783, 'group_types': {'single': 1020477, 'same_host_diff_path': 1665, 'url_variant': 1576, 'cross_host': 65}, 'sample_docs_per_part': 0, 'languages': {'en': {'texts': 239775, 'doc_ids': 239918, 'chunks': 250487, 'texts_changed_by_cleaning': 7, 'doc_ids_changed_by_cleaning': 8, 'chunks_not_in_raw': 7, 'chunks_per_text_mean': 1.04, 'chunks_per_text_p50': 1, 'chunks_per_text_p90': 1, 'chunks_per_text_max': 6, 'n_units_p50': 201, 'n_units_p90': 298, 'n_units_max': 435, 'read_seconds': 70.0, 'chunk_seconds': 14.2, 'write_seconds': 2.4, 'chunk_seconds_per_1000_texts': 0.06, 'seconds_per_1000_texts': 0.36}, 'vi': {'texts': 685354, 'doc_ids': 688537, 'chunks': 2128601, 'texts_changed_by_cleaning': 0, 'doc_ids_changed_by_cleaning': 0, 'chunks_not_in_raw': 0, 'chunks_per_text_mean': 3.11, 'chunks_per_text_p50': 3, 'chunks_per_text_p90': 5, 'chunks_per_text_max': 383, 'n_units_p50': 320, 'n_units_p90': 370, 'n_units_max': 438, 'read_seconds': 385.6, 'chunk_seconds': 248.0, 'write_seconds': 32.0, 'chunk_seconds_per_1000_texts': 0.36, 'seconds_per_1000_texts': 0.97}, 'zh': {'texts': 98654, 'doc_ids': 98943, 'chunks': 242465, 'texts_changed_by_cleaning': 34564, 'doc_ids_changed_by_cleaning': 34706, 'chunks_not_in_raw': 68, 'chunks_per_text_mean': 2.46, 'chunks_per_text_p50': 1, 'chunks_per_text_p90': 3, 'chunks_per_text_max': 2454, 'n_units_p50': 256, 'n_units_p90': 352, 'n_units_max': 978, 'read_seconds': 36.7, 'chunk_seconds': 8.0, 'write_seconds': 2.1, 'chunk_seconds_per_1000_texts': 0.08, 'seconds_per_1000_texts': 0.47}}, 'elapsed_seconds': 840.5}, 'release_manifest_sha256': '1e6e2e710e6355d769d8fad7cc57e21c9400d6d029daab91058438d5f3e8f88f', 'old_chunk_paths': ['chunks/en/part-000.parquet', 'chunks/vi/part-000.parquet', 'chunks/vi/part-001.parquet', 'chunks/vi/part-002.parquet', 'chunks/vi/part-003.parquet', 'chunks/vi/part-004.parquet', 'chunks/vi/part-005.parquet', 'chunks/vi/part-006.parquet', 'chunks/zh/part-000.parquet'], 'old_chunks': 2621553}
SOURCE='"""Kaggle CPU: preserve baseline parts and append verified language partitions."""\nfrom pathlib import Path\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport hashlib,json,shutil,zipfile\nimport pyarrow as pa\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as handle:return hashlib.file_digest(handle,\'sha256\').hexdigest()\n\n\ndef package(baseline,release,out,spec):\n    baseline,release,out=map(Path,(baseline,release,out));out.mkdir(parents=True,exist_ok=True)\n    manifest=json.loads((baseline/\'chunk_manifest.json\').read_text())\n    assert manifest==spec[\'baseline_manifest\']\n    assert sha(release/\'release_manifest.json\')==spec[\'release_manifest_sha256\']\n    rm=json.loads((release/\'release_manifest.json\').read_text())\n    for name in [\'chunk_corpus_additions_20261029_v2.parquet\',\'hash_groups_additions_20261029_v2.parquet\',\'aliases_to_existing_20261029_v2.parquet\',\'documents_additions_20261029_v2.parquet\']:\n        assert sha(release/name)==rm[\'files\'][name][\'sha256\']\n    parts=sorted(baseline.glob(\'chunks/*/part-*.parquet\'))\n    assert [p.relative_to(baseline).as_posix() for p in parts]==spec[\'old_chunk_paths\']\n    ids=set();base_counts=Counter();base_schemas={};old_files={}\n    for path in parts:\n        rel=path.relative_to(baseline);target=out/rel;target.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copyfile(path,target);digest=sha(path);assert sha(target)==digest\n        old_files[rel.as_posix()]=dict(sha256=digest,bytes=path.stat().st_size)\n        lang=path.parent.name;base_schemas[lang]=pq.read_schema(path)\n        for batch in pq.ParquetFile(path).iter_batches(columns=[\'chunk_id\',\'lang\']):\n            current=batch.column(0).to_pylist();assert len(set(current))==len(current) and not ids.intersection(current)\n            assert set(batch.column(1).to_pylist())=={lang}\n            ids.update(current);base_counts[lang]+=len(current)\n    assert sum(base_counts.values())==spec[\'old_chunks\']\n    additions=pq.read_table(release/\'chunk_corpus_additions_20261029_v2.parquet\')\n    added_ids=additions[\'chunk_id\'].to_pylist()\n    assert len(set(added_ids))==len(added_ids)==rm[\'new_chunks\'] and not ids.intersection(added_ids)\n    new_counts={};new_paths={}\n    for lang in [\'vi\',\'en\',\'zh\']:\n        current=additions.filter(pc.equal(additions[\'lang\'],lang))\n        assert len(current)==rm[\'chunks_by_language\'][lang]\n        previous=[int(p.stem.split(\'-\')[-1]) for p in parts if p.parent.name==lang]\n        rel=f\'chunks/{lang}/part-{max(previous)+1:03d}.parquet\';target=out/rel\n        assert not target.exists()\n        current=current.select(base_schemas[lang].names).cast(base_schemas[lang].remove_metadata())\n        pq.write_table(current,target,compression=\'zstd\')\n        saved=pq.read_table(target)\n        assert saved.equals(current) and saved[\'chunk_id\'].to_pylist()==current[\'chunk_id\'].to_pylist()\n        new_counts[lang]=len(current);new_paths[lang]=rel\n        print(f\'Added {rel}: {len(current):,} rows\',flush=True)\n    old_groups=pq.read_table(baseline/\'hash_groups.parquet\')\n    new_groups=pq.read_table(release/\'hash_groups_additions_20261029_v2.parquet\')\n    old_hashes=set(old_groups[\'text_sha256\'].to_pylist());new_hashes=set(new_groups[\'text_sha256\'].to_pylist())\n    assert len(old_hashes)==len(old_groups) and len(new_hashes)==len(new_groups) and not old_hashes.intersection(new_hashes)\n    new_groups=new_groups.select(old_groups.schema.names).cast(old_groups.schema.remove_metadata())\n    old_plain=old_groups.replace_schema_metadata(None)\n    combined=pa.concat_tables([old_plain,new_groups]);pq.write_table(combined,out/\'hash_groups.parquet\',compression=\'zstd\')\n    assert pq.read_table(out/\'hash_groups.parquet\').slice(0,len(old_groups)).equals(old_plain)\n    old_doc_ids=set();new_doc_ids=set()\n    for group in old_groups.select([\'doc_ids\',\'n_ids\']).to_pylist():\n        members=group[\'doc_ids\'];assert len(set(members))==len(members)==group[\'n_ids\'] and not old_doc_ids.intersection(members)\n        old_doc_ids.update(members)\n    for group in new_groups.select([\'doc_ids\',\'n_ids\']).to_pylist():\n        members=group[\'doc_ids\'];assert len(set(members))==len(members)==group[\'n_ids\'] and not new_doc_ids.intersection(members) and not old_doc_ids.intersection(members)\n        new_doc_ids.update(members)\n    doc_lang={r[\'doc_id\']:r[\'lang\'] for r in pq.read_table(release/\'documents_additions_20261029_v2.parquet\',columns=[\'doc_id\',\'lang\']).to_pylist()}\n    added_doc_counts=Counter(doc_lang[i] for i in new_doc_ids)\n    added_text_counts=Counter(doc_lang[i] for i in new_groups[\'rep_doc_id\'].to_pylist())\n    pending=pq.read_table(release/\'aliases_to_existing_20261029_v2.parquet\')\n    assert len(pending)==rm[\'alias_document_ids\'] and len(new_doc_ids)+len(pending)==rm[\'accepted_document_ids\']\n    (out/\'metadata\').mkdir();(out/\'provenance\').mkdir()\n    shutil.copyfile(release/\'aliases_to_existing_20261029_v2.parquet\',out/\'metadata/aliases_pending_review.parquet\')\n    shutil.copyfile(baseline/\'chunk_manifest.json\',out/\'provenance/chunk_manifest_v1.json\')\n    shutil.copyfile(release/\'release_manifest.json\',out/\'provenance/additions_release_manifest.json\')\n    result=dict(dataset_version=\'v2\',dataset_built_at=datetime.now(timezone.utc).isoformat(),\n        clean_version=manifest[\'clean_version\'],chunk_params=manifest[\'chunk_params\'],id_scheme=manifest[\'id_scheme\'],\n        excluded_ids=manifest[\'excluded_ids\'],hash_groups=len(combined),group_types=dict(Counter(combined[\'group_type\'].to_pylist())),\n        languages={lang:dict(texts=manifest[\'languages\'][lang][\'texts\']+added_text_counts[lang],\n            doc_ids=manifest[\'languages\'][lang][\'doc_ids\']+added_doc_counts[lang],chunks=base_counts[lang]+new_counts[lang]) for lang in [\'vi\',\'en\',\'zh\']},\n        total_chunks=len(ids)+len(added_ids),chunk_parts=len(parts)+3,baseline_chunk_parts_unchanged=True,\n        additions=dict(document_ids=rm[\'accepted_document_ids\'],indexed_document_ids=len(new_doc_ids),\n            pending_alias_document_ids=len(pending),new_chunks=len(added_ids),parts=new_paths,\n            release_manifest_sha256=spec[\'release_manifest_sha256\'],source_kernel=\'ntklinhfitus/medxir-recovery-finalize-20261029-v2\'),\n        baseline_chunk_files=old_files,embedding_changes=False)\n    (out/\'chunk_manifest.json\').write_text(json.dumps(result,indent=2))\n    report=dict(complete=True,total_chunks=result[\'total_chunks\'],new_chunks=len(added_ids),chunk_parts=result[\'chunk_parts\'],\n        old_chunk_files_unchanged=True,old_group_rows_unchanged=True,chunk_ids_unique=True,chunk_ids_preserved=True,\n        old_chunks=len(ids),languages=result[\'languages\'],pending_alias_document_ids=len(pending),\n        indexed_document_ids=len(old_doc_ids)+len(new_doc_ids),new_indexed_document_ids=len(new_doc_ids),\n        files={p.relative_to(out).as_posix():dict(bytes=p.stat().st_size,sha256=sha(p)) for p in sorted(out.rglob(\'*\')) if p.is_file()})\n    return report\n\n\ndef main():\n    spec=json.loads(Path(\'/kaggle/working/chunk_update_spec.json\').read_text())\n    base=list(Path(\'/kaggle/input\').rglob(\'chunk_manifest.json\'));releases=list(Path(\'/kaggle/input\').rglob(\'release_manifest.json\'))\n    assert len(base)==len(releases)==1\n    payload=Path(\'/kaggle/working/dataset_payload\')\n    report=package(base[0].parent,releases[0].parent,payload,spec)\n    archive=Path(\'/kaggle/working/dataset_payload.zip\')\n    with zipfile.ZipFile(archive,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n        for p in sorted(payload.rglob(\'*\')):\n            if p.is_file():z.write(p,p.relative_to(payload).as_posix())\n    report[\'archive\']=dict(bytes=archive.stat().st_size,sha256=sha(archive))\n    Path(\'/kaggle/working/chunk_dataset_update_manifest.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps({k:v for k,v in report.items() if k!=\'files\'},indent=2))\n\n\nif __name__==\'__main__\':main()\n'
from pathlib import Path
import json,subprocess,sys
p=Path('/kaggle/working/package_chunk_dataset_update.py');p.write_text(SOURCE,encoding='utf-8')
Path('/kaggle/working/chunk_update_spec.json').write_text(json.dumps(SPEC))
subprocess.run([sys.executable,'-u',str(p)],check=True)
